# LO2v2 data preprocessing

Build reusable datasets for the ML and deep learning phases. The pipeline parses Unix timestamps as UTC, joins seven services on the same run, scenario and timestamp, and converts cumulative counters to per-second rates. Rates are missing at scenario starts, after gaps or counter resets. Exact duplicates are removed and conflicting keys are reported.

The pipeline uses 20-second windows with a 10-second stride and requires all seven services to have finite features at every second. Windows stay within their scenario. Each service contributes mean and standard deviation for seven signals. The command below writes aligned time series, split feature tables and scaler parameters.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
completed = subprocess.run(
    [sys.executable, "-m", "src.preprocess"], cwd=root,
    check=True, capture_output=True, text=True,
)
print(completed.stdout)

output = root / "data" / "processed"
metadata = json.loads((output / "preprocessing.json").read_text())
print(f"Aligned rows: {metadata['aligned_rows']:,}")
print(f"Aligned rows with unavailable signals: {metadata['aligned_incomplete_rows']:,}")

error       False  True 
split                   
test           11    823
train          54   2462
validation      6    851

Aligned rows: 55,317
Aligned rows with unavailable signals: 541


## Verify split integrity and check training normalization

The split assigns entire runs to training, validation or testing, so overlapping windows from a run cannot cross the split. The scaler learns mean and variance from training windows. Check that feature values are finite in every split, every window spans 20 seconds inclusive, and the training feature means are approximately zero.

In [2]:
tables = {
    split: pd.read_parquet(output / f"windows_{split}.parquet")
    for split in metadata["runs"]
}
features = metadata["feature_columns"]
all_runs = [run for members in metadata["runs"].values() for run in members]
assert len(all_runs) == len(set(all_runs))

for split, table in tables.items():
    assert set(table.run) == set(metadata["runs"][split])
    assert table.split.eq(split).all()
    assert (table.window_end - table.window_start).dt.total_seconds().eq(19).all()
    assert np.isfinite(table[features].to_numpy()).all()

assert np.allclose(tables["train"][features].mean(), 0)
display(pd.DataFrame({
    split: {"Runs": data.run.nunique(), "Windows": len(data),
            "Normal": int((~data.error).sum()), "Error": int(data.error.sum())}
    for split, data in tables.items()
}).T)
display(tables["train"][["run", "test", "window_start", "window_end", *features[:2]]].head())

,Runs,Windows,Normal,Error
train,6,2516,54,2462
validation,2,857,6,851
test,2,834,11,823


,run,test,window_start,window_end,client__container_memory_rss__mean,client__container_memory_rss__std
0,LO2_run_1739613601,access_token_auth_header_error_401,2025-02-15 10:04:12+00:00,2025-02-15 10:04:31+00:00,-0.610690,-0.106618
1,LO2_run_1739613601,access_token_auth_header_error_401,2025-02-15 10:04:22+00:00,2025-02-15 10:04:41+00:00,-0.609252,-0.095376
2,LO2_run_1739613601,access_token_auth_header_error_401,2025-02-15 10:04:32+00:00,2025-02-15 10:04:51+00:00,-0.608275,-0.100192
3,LO2_run_1739613601,access_token_auth_header_error_401,2025-02-15 10:04:42+00:00,2025-02-15 10:05:01+00:00,-0.607664,-0.105381
4,LO2_run_1739613601,access_token_auth_header_error_401,2025-02-15 10:04:52+00:00,2025-02-15 10:05:11+00:00,-0.606412,-0.104317
